In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Glow Lab's stock feed: JSON ➜ Apache Iceberg with PySpark

Glow Lab Beauty, YeYo's marketplace partner, drops a JSON stock feed in Cloud Storage. This notebook reads it with
**PySpark on Serverless for Apache Spark** (no cluster to manage) and appends it to the Apache Iceberg table
`yeyo_lake.yeyo_partner_lake.partner_stock` through the BigLake Iceberg REST catalog. BigQuery then queries the same
Iceberg files, with no copy.

**Packaged as a job:** the ingestion cell (section 4) is the same code as
[`spark/partner_stock_sync.py`](https://github.com/ak777app/retail-data-to-ai-workshop/blob/main/02-code-assets/yeyo-storefront/spark/partner_stock_sync.py).
The storefront's **Reset demo** submits that file as a Serverless Spark batch job with the new feed, so the notebook and
the job can't drift apart. Setup is Module 13 sections 1-3.

## 1. Settings

In [ ]:
%pip install -q dataproc-spark-connect

In [ ]:
import google.auth
from google.auth.transport.requests import AuthorizedSession

PROJECT_ID = ""  # leave empty to use the notebook's own project
credentials, default_project = google.auth.default()
PROJECT_ID = PROJECT_ID or default_project
PROJECT_NBR = AuthorizedSession(credentials).get(
    f"https://cloudresourcemanager.googleapis.com/v3/projects/{PROJECT_ID}").json()["name"].split("/")[1]

LOCATION = "us-central1"
BUCKET = ""  # the storefront's LAKEHOUSE_BUCKET; leave empty for Module 13's yeyo-lakehouse-<project number>
BUCKET = BUCKET or f"yeyo-lakehouse-{PROJECT_NBR}"
SPARK_SA = f"{PROJECT_NBR}-compute@developer.gserviceaccount.com"  # the identity Reset demo's batch job runs as
CATALOG, NAMESPACE, TABLE = "yeyo_lake", "yeyo_partner_lake", "partner_stock"
TABLE_FQ = f"{CATALOG}.{NAMESPACE}.{TABLE}"
print(PROJECT_ID, BUCKET, TABLE_FQ)

## 2. The partner's JSON feed

The newest feed in `gs://BUCKET/partner_feed/glowlab/`: one JSON record per listing. **Reset demo** drops a new one each
time. With no feed yet, section 4 generates the same three records (600 units for the viral listing `LST-0017`).

In [ ]:
from google.cloud import storage

feeds = sorted(storage.Client(project=PROJECT_ID).list_blobs(BUCKET, prefix="partner_feed/glowlab/"),
               key=lambda b: b.updated)
FEED = f"gs://{BUCKET}/{feeds[-1].name}" if feeds else None
print(FEED or "No feed yet")
print(feeds[-1].download_as_text() if feeds else "")

## 3. A Serverless Spark session with the Iceberg REST catalog

Spark runs on Serverless for Apache Spark through Spark Connect. The catalog properties are the ones **Reset demo**'s
batch job uses (`storefront/ops.py`, `restock_batch`), and Spark runs as the same service account. The session takes
about a minute to start.

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

C, JARS = f"spark.sql.catalog.{CATALOG}", f"gs://{BUCKET}/jars"
session = Session()
session.runtime_config.version = "3.0"
session.environment_config.execution_config.service_account = SPARK_SA
session.runtime_config.properties = {
    "spark.jars": f"{JARS}/iceberg-spark-runtime-4.0_2.13-1.10.2.jar,{JARS}/iceberg-gcp-bundle-1.10.2.jar",
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    C: "org.apache.iceberg.spark.SparkCatalog",
    f"{C}.type": "rest",
    f"{C}.uri": "https://biglake.googleapis.com/iceberg/v1/restcatalog",
    f"{C}.warehouse": f"bl://projects/{PROJECT_ID}/catalogs/{CATALOG}",
    f"{C}.header.x-goog-user-project": PROJECT_ID,
    f"{C}.rest.auth.type": "org.apache.iceberg.gcp.auth.GoogleAuthManager",
    f"{C}.io-impl": "org.apache.iceberg.gcp.gcs.GCSFileIO",
    "spark.dataproc.lineage.enabled": "true",
}
spark = (DataprocSparkSession.builder.appName("glowlab-partner-stock-notebook")
         .projectId(PROJECT_ID).location(LOCATION).dataprocSessionConfig(session).getOrCreate())

In [ ]:
if FEED:
    spark.read.json(FEED).show(truncate=False)

## 4. Ingest: JSON ➜ Iceberg

This cell is the shared block of `spark/partner_stock_sync.py`, the job **Reset demo** runs. Plain PySpark: read the
JSON with a fixed schema, then `writeTo(...).append()` adds a new Iceberg snapshot.

In [ ]:
from datetime import datetime, timezone

LISTINGS = [
    ("LST-0017", "MKT-S004", "BEA-3001-OM", "BEA-3001"),
    ("LST-0018", "MKT-S004", "BEA-3002-OM", "BEA-3002"),
    ("LST-0019", "MKT-S004", "BEA-3003-OM", "BEA-3003"),
]
SCHEMA = ("listing_id string, seller_id string, omni_item_id string, item_number string, "
          "warehouse_id string, units_on_hand int, snapshot_ts timestamp, source_system string")


def ingest_partner_stock(spark, table, feed=None, units=600, other_units=1500):
    """Appends one stock snapshot per listing to the Iceberg table, read from the partner's JSON feed if given."""
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {table.rsplit('.', 1)[0]}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
          listing_id STRING, seller_id STRING, omni_item_id STRING, item_number STRING,
          warehouse_id STRING, units_on_hand INT, snapshot_ts TIMESTAMP, source_system STRING
        ) USING iceberg
    """)
    if feed:
        df = spark.read.schema(SCHEMA).json(feed)
    else:
        now = datetime.now(timezone.utc).replace(tzinfo=None)
        rows = [(lid, sid, omni, item, "GLOWLAB-EVANSTON-DC", units if lid == "LST-0017" else other_units,
                 now, "glowlab-spark") for lid, sid, omni, item in LISTINGS]
        df = spark.createDataFrame(rows, SCHEMA)
    df.writeTo(table).append()
    return spark.table(table).orderBy("snapshot_ts", ascending=False)

In [ ]:
latest = ingest_partner_stock(spark, TABLE_FQ, feed=FEED)
latest.show(6, truncate=False)

## 5. BigQuery reads the same Iceberg table

No load, no copy: BigQuery queries the files Spark just wrote, by the catalog's four-part name.

In [ ]:
import pandas as pd
from google.cloud import bigquery

sql = (f"SELECT listing_id, units_on_hand, snapshot_ts, source_system "
       f"FROM `{PROJECT_ID}.{CATALOG}.{NAMESPACE}.{TABLE}` ORDER BY snapshot_ts DESC LIMIT 6")
pd.DataFrame([dict(r) for r in bigquery.Client(project=PROJECT_ID, location=LOCATION).query(sql).result()])

## 6. The same code as a job

**Reset demo** uploads a new feed and submits `spark/partner_stock_sync.py` as a Serverless Spark batch, roughly:

```
gcloud dataproc batches submit pyspark gs://BUCKET/spark/partner_stock_sync.py --region=us-central1 --version=2.3 \
  --jars=gs://BUCKET/jars/iceberg-spark-runtime-3.5_2.13-1.10.2.jar,gs://BUCKET/jars/iceberg-gcp-bundle-1.10.2.jar \
  --properties=... -- --feed gs://BUCKET/partner_feed/glowlab/stock-....json
```

The runs are listed under **Dataproc ➜ Serverless ➜ Batches**. After changing the shared block, copy it into both
files (`tests/test_platform.py` checks they match) and re-upload the script:
`gcloud storage cp spark/partner_stock_sync.py gs://BUCKET/spark/`.

In [ ]:
spark.stop()